# 📣 หน่วยที่ 6: Telegram Integration

บทนี้จะสอนวิธีการใช้ Python และเครื่องมือมาตรฐาน (ไลบรารี `requests`) ในการส่งข้อความและรูปภาพไปยัง **Telegram** โดยเราจะสามารถนำฟังก์ชันเหล่านี้ไปเสียบเข้ากับไลบรารีของ Agent ที่มีอยู่เพื่อให้มันสามารถแจ้งเตือนมายังมือถือเราได้ครับ

## 📌 สิ่งที่ต้องเตรียม (ตั้งค่าใน BotFather)

1. **สร้าง Bot**: ค้นหาบัญชี `@BotFather` ใน Telegram, พิมพ์ `/newbot`, ตั้งชื่อและ username เพื่อรับ **`TELEGRAM_BOT_TOKEN`**
2. **หา Chat ID**: ลองส่งข้อความหาบอทที่เพิ่งสร้าง แล้วเข้า URL: `https://api.telegram.org/bot<YOUR_TOKEN>/getUpdates` เพื่อรับและคัดลอกค่า `chat` -> `id` มาเป็น **`TELEGRAM_CHAT_ID`**
3. เพิ่มค่า Token และ Chat ID ลงในแค็ตตาล็อก `.env` ของคุณ

> **ตัวอย่าง .env**:
> ```
> TELEGRAM_BOT_TOKEN=123456789:ABCDEfghij...
> TELEGRAM_CHAT_ID=1234567
> ```

In [ ]:
import os
import requests
from dotenv import load_dotenv

# โหลดค่า Token และ Chat ID จากไฟล์ .env
load_dotenv()

TELEGRAM_BOT_TOKEN = os.getenv('TELEGRAM_BOT_TOKEN')
TELEGRAM_CHAT_ID = os.getenv('TELEGRAM_CHAT_ID')

if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
    print("⚠️ ยังไม่ได้ตั้งค่า TELEGRAM_BOT_TOKEN หรือ TELEGRAM_CHAT_ID ในไฟล์ .env")
else:
    print("✅ พร้อมส่งข้อความ! (โหลดค่า Token & Chat ID เรียบร้อย)")

## ✉️ การส่งข้อความปกติ (Send Message)

In [ ]:
def send_telegram_message(message: str):
    """ฟังก์ชันพื้นฐานสำหรับส่งข้อความไปยัง Telegram Chat"""
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        return False
    
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {
        "chat_id": TELEGRAM_CHAT_ID,
        "text": message,
        "parse_mode": "HTML" # ให้ใช้รูปแบบ Markdown (ทำตัวหนา หนา-เอียง ได้)
    }
    
    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            print("✅ ส่งข้อความสำเร็จ!")
            return True
        else:
            print(f"❌ ส่งข้อความล้มเหลว: {response.text}")
            return False
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

# ลองส่งข้อความทดสอบ
test_msg = "*ทดสอบแจ้งเตือนจาก Notebook!*\
\\ระบบ Telegram Integration สำเร็จ 🚀"
send_telegram_message(test_msg)

## 🖼️ การส่งรูปภาพ (Send Photo)
การวิเคราะห์กราฟ Top-Down หรือ SMC สำคัญที่รูปภาพ เราสามารถส่งรูปแบบ Base64 (ที่ Agent ใช้) ไป Telegram ได้เลย แต่ต้องแปลงเป็นรูปแบบไฟล์ (BytesIO) ก่อนเพื่อให้ Telegram เข้าใจ

In [ ]:
import io
import base64
import requests

def send_telegram_base64_photo(base64_str: str, caption: str = ""):
    """แปลงรูปภาพแบบ Base64 (เช่น data:image/png;base64,...) และส่งเป็นไฟล์รูปไปยัง Telegram"""
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        return False
        
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendPhoto"
    
    try:
        # เช็คหากมีส่วน data URL ให้ตัดทิ้ง
        if "," in base64_str:
            base64_str = base64_str.split(",")[1]
            
        # แปลง string base64 เป็น byte stream
        image_data = base64.b64decode(base64_str)
        image_file = io.BytesIO(image_data)
        image_file.name = "chart.png" # ตั้งชื่อไฟล์จำลองให้ telegram API ยอมรับ
        
        # การส่งไฟล์ทาง requests (multipart/form-data)
        files = {"photo": image_file}
        data = {"chat_id": TELEGRAM_CHAT_ID, "caption": caption, "parse_mode": "HTML"}
        
        response = requests.post(url, data=data, files=files)
        if response.status_code == 200:
            print("✅ ส่งรูปภาพสำเร็จ!")
            return True
        else:
            print(f"❌ ส่งรูปภาพล้มเหลว: {response.text}")
            return False
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

### 🧪 ทดลองสร้างรูปและส่ง

In [ ]:
import matplotlib.pyplot as plt

# สร้างกราฟง่ายๆ แบบจำลอง
plt.figure(figsize=(6, 4))
plt.plot([1, 2, 3, 4, 5], [10, 15, 13, 20, 25], label="Price Action")
plt.title("Sample Output Chart")
plt.legend()

# จับกราฟลงใน BytesIO และแปลงเป็น Base64
buf = io.BytesIO()
plt.savefig(buf, format='png', bbox_inches='tight')
buf.seek(0)
sample_b64 = base64.b64encode(buf.getvalue()).decode()
plt.close()

# ลองส่งกราฟ
send_telegram_base64_photo(sample_b64, caption="📊 นี่คือกราฟตัวอย่างที่ระบบเราจะส่งเข้าไปใน 05_full_trading_agent")

## 📁 การส่งรูปจากไฟล์ในฮาร์ดดิสก์ตรงๆ (Direct Physical File)

หากเรามีการเซฟไฟล์กราฟเก็บไว้ในโฟลเดอร์อยู่แล้ว (เช่น `chart.png` หรือ `my_trade.jpg`) เราสามารถสั่งให้ Python เปิดไฟล์นั้นจากเครื่องแล้วส่งอัปโหลดไปให้ Telegram ตรงๆ ได้เลย โดยไม่ต้องพึ่ง Base64 ครับ!

In [ ]:
def send_telegram_file_directly(file_path: str, caption: str = ""):
    """ฟังก์ชันสำหรับอ่านไฟล์รูปจากเครื่อง และส่งไปยัง Telegram ทันที"""
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        return False
        
    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendPhoto"
    
    try:
        # เปิดไฟล์จากเครื่องแบบ 'rb' (Read Binary - อนุญาตให้อ่านข้อมูลไบนารี)
        with open(file_path, 'rb') as photo_file:
            files = {"photo": photo_file}
            data = {"chat_id": TELEGRAM_CHAT_ID, "caption": caption, "parse_mode": "HTML"}
            
            response = requests.post(url, data=data, files=files)
            if response.status_code == 200:
                print(f"✅ ส่งไฟล์ {file_path} สำเร็จ!")
                return True
            else:
                print(f"❌ ส่งไฟล์ล้มเหลว: {response.text}")
                return False
                
    except FileNotFoundError:
        print(f"❌ หาไฟล์ '{file_path}' ไม่เจอครับ กรุณาเช็คตัวสะกดและโฟลเดอร์")
        return False
    except Exception as e:
        print(f"❌ Error: {e}")
        return False

# ================================
# 🧪 ทดสอบเซฟเป็นไฟล์แล้วส่งดู!
# ================================
import matplotlib.pyplot as plt
import os

# 1. วาดรูปแล้วเซฟลงคอมจริงๆ
plt.figure(figsize=(4, 3))
plt.plot([5, 4, 3, 2, 1], [10, 5, 20, 15, 30], color='red', marker='o')
plt.title("File from Hard Disk")
file_name = "test_physical_chart.png"
plt.savefig(file_name)
plt.close()

# 2. ลองส่งไฟล์
send_telegram_file_directly(file_name, caption="📁 นี่คือรูปที่ถูกเซฟลงคอมและอัปโหลดจากฮาร์ดดิสก์ของคุณโดยตรงครับ!")

# (optional) เผื่อใจดี ลบไฟล์ขยะทิ้งให้หลังส่งเสร็จ:
# if os.path.exists(file_name):
#     os.remove(file_name)


---
## 🔗 การเตรียมตัวนำไปใช้ใน 05_full_trading_agent.ipynb
ใน Notebook ที่ 5 เราสามารถครอบโค้ด `topdown_analysis()` และ `auto_loop()` โดย:

1. **แปะฟังก์ชัน `send_telegram_message()` และ `send_telegram_base64_photo()`** ไว้ที่เซลล์ช่วงล่างๆ
2. ตอนที่ **Top-Down Analysis** สร้าง result และภาพ Base64 เสร็จแล้ว ก็ให้ใช้ลูป `for` สั่ง `send_telegram_base64_photo` ต่อเลย
3. ตอนจบแต่ละรอบของ **Auto Loop** ก็สามารถสั่ง `send_telegram_message(r['output'])` ได้เช่นกันครับ!